# 仓库记忆和持久化状态

会话历史是易变的，但是仓库是持久的。工作台将agent状态存进带版本的文件，然后下一个会话、下一个agent、下一个审查者都能够从相同的事实源码中读取。

## 问题描述

agent完成了一个会话，然后关闭了。下一会话打开，然后询问从哪来开始。模型表示“让我先检查一下文件”，开始读取状态记录，然后把已经完成的工作又做了一遍。或者更糟，它把一个已经完成的文件重写了一遍，因为没有人告诉它这个文件已经完成了。

工作台修复的方式是仓库记忆：状态通过JSON文件的方式记录在仓库中，以符合schema的形式写入，自动持久化，而且在代码审核的时候差异友好。聊天只是瞬时投喂，仓库才是记录系统。

## 基本概念

```mermaid
flowchart LR
A[Agent Loop] --> B[State Manager] --> C[agent_state.schema.json] --> D(Valid?)
D --yes --> E[agent_state.json]
D --no -->F[refues + raise]
E -->B
```

### 哪些东西属于仓库记忆

|属于|不属于|
|---|---|
|激活任务id|原本的聊天记录|
|会话中动到的文件|token层级的推理轨迹|
|agent做出的假设|对用户的情感推测|
|blockers，缺权限、依赖挂了等，下一轮需要审查者看见|抽样完成，这类瞬时过程结论，不属于仓库里的事实|
|下一个行动|供应商特定的模型id|

区分原则是持久性：这个东西是否在3个月后的某次CI重试中生效？是，就是仓库持久记忆。否则，临时记忆。

### SCHEMA-FIRST STATE

JSON Schema 是契约。没有它，每个agent都在发明新的字段，每个审查者都需要学习新的形态，CI脚本需要为过往的版本作特化控制。有了它，一次坏的写将被拒绝。

Schema 包含:
1. 必要的键值
2. 数值范围限制（可行的值、禁止的值）
3. 模式限制（正则匹配）
4. 迁移所需的版本控制

### 自动写

状态写入需要应该部分失败：写入了临时文件、磁盘同步、重命名。状态文件是说明事实的源码，写了一半比完全没有更糟。

### 迁移

当schema 变更的时候，同时上线迁移脚本。状态文件包含`schema_version`字段，状态管理器拒绝加载来自无法迁移版本的状态文件。

# 开始编码

对应本章核心：**仓库记忆 = 带 schema 的 `agent_state.json`**、**属于/不属于过滤**、**原子写（tmp→fsync→rename）**、**`schema_version` + 迁移**。  
玩具先跑通 StateManager；生产段用 **LangChain 工具 + DeepSeek** 强制经工具读写状态。不硬凑 PyTorch。


## 1. 教学玩具：Schema-first StateManager

- JSON Schema 拒坏写；瞬时字段（聊天 / sampling_done / model id）写入前过滤。
- 原子写：临时文件 + `fsync` + `os.replace`。
- `schema_version`：v1→v2 迁移；未知版本直接拒绝。


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import tempfile
import threading
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable

from jsonschema import Draft202012Validator, ValidationError


CURRENT_SCHEMA_VERSION = 2

# 仓库记忆契约：字段变更必须抬 schema_version 并提供迁移。
AGENT_STATE_SCHEMA: dict[str, Any] = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "additionalProperties": False,
    "required": [
        "schema_version",
        "active_task_id",
        "touched_files",
        "assumptions",
        "blockers",
        "next_action",
    ],
    "properties": {
        "schema_version": {"type": "integer", "minimum": 1},
        "active_task_id": {
            "oneOf": [
                {"type": "null"},
                {"type": "string", "pattern": "^[A-Z0-9_-]+$"},
            ]
        },
        "touched_files": {
            "type": "array",
            "items": {"type": "string", "minLength": 1},
            "uniqueItems": True,
        },
        "assumptions": {"type": "array", "items": {"type": "string", "minLength": 1}},
        "blockers": {"type": "array", "items": {"type": "string", "minLength": 1}},
        "next_action": {"type": "string"},
        # v2 新增：跨会话可读的简短进度摘要（仍属事实，不含聊天）
        "progress_note": {"type": "string", "maxLength": 200},
    },
}


@dataclass
class AgentState:
    """持久化仓库记忆：只放跨会话仍有效的事实。"""

    schema_version: int = CURRENT_SCHEMA_VERSION
    active_task_id: str | None = None
    touched_files: list[str] = field(default_factory=list)
    assumptions: list[str] = field(default_factory=list)
    blockers: list[str] = field(default_factory=list)
    next_action: str = ""
    progress_note: str = ""

    def to_dict(self) -> dict[str, Any]:
        """
        Returns:
            payload: 可校验、可落盘的 dict。
        """
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "AgentState":
        """
        Args:
            data: 已通过 schema 校验的 payload。
        Returns:
            state: AgentState 实例。
        """
        return cls(
            schema_version=int(data["schema_version"]),
            active_task_id=data.get("active_task_id"),
            touched_files=list(data.get("touched_files", [])),
            assumptions=list(data.get("assumptions", [])),
            blockers=list(data.get("blockers", [])),
            next_action=str(data.get("next_action", "")),
            progress_note=str(data.get("progress_note", "")),
        )


# 临时记忆 vs 仓库记忆：3 个月后 CI 还需要吗？
EPHEMERAL_KEYS = {
    "chat_transcript",
    "token_trace",
    "user_emotion",
    "sampling_done",
    "model_vendor_id",
}
REPO_KEYS = {
    "active_task_id",
    "touched_files",
    "assumptions",
    "blockers",
    "next_action",
    "progress_note",
    "schema_version",
}


def filter_repo_memory(raw: dict[str, Any]) -> dict[str, Any]:
    """
    只保留属于仓库记忆的键；瞬时字段丢弃。

    Args:
        raw: 混合了聊天/过程噪声的草稿。
    Returns:
        cleaned: 可写入 agent_state.json 的子集。
    """
    return {k: v for k, v in raw.items() if k in REPO_KEYS and k not in EPHEMERAL_KEYS}


def migrate_state(data: dict[str, Any]) -> dict[str, Any]:
    """
    按 schema_version 链式迁移到 CURRENT。

    Args:
        data: 磁盘上的原始 JSON。
    Returns:
        migrated: 当前版本 payload（尚未最终校验）。
    Raises:
        ValueError: 无法识别或无法迁移的版本。
    """
    payload = deepcopy(data)
    version = int(payload.get("schema_version", 0))
    if version < 1 or version > CURRENT_SCHEMA_VERSION:
        raise ValueError(f"unsupported schema_version={version}")

    if version == 1:
        # v1: current_task + files → v2: active_task_id + touched_files + progress_note
        if "active_task_id" not in payload and "current_task" in payload:
            payload["active_task_id"] = payload.pop("current_task")
        if "touched_files" not in payload and "files" in payload:
            payload["touched_files"] = payload.pop("files")
        payload.setdefault("assumptions", [])
        payload.setdefault("blockers", [])
        payload.setdefault("next_action", "")
        payload.setdefault("progress_note", "")
        payload["schema_version"] = 2
        version = 2

    if version != CURRENT_SCHEMA_VERSION:
        raise ValueError(f"migration stalled at schema_version={version}")
    return payload


class StateManager:
    """Schema-first + 原子写的仓库记忆管理器。"""

    def __init__(self, root: Path, *, schema: dict[str, Any] | None = None) -> None:
        """
        Args:
            root: 仓库根目录。
            schema: JSON Schema；默认 AGENT_STATE_SCHEMA。
        """
        self.root = Path(root)
        self.path = self.root / "agent_state.json"
        self.schema = schema or AGENT_STATE_SCHEMA
        self._validator = Draft202012Validator(self.schema)
        self._lock = threading.Lock()

    def validate(self, data: dict[str, Any]) -> None:
        """
        Args:
            data: 待校验 payload。
        Raises:
            ValidationError: schema 拒绝坏写入。
        """
        self._validator.validate(data)

    def atomic_write(self, data: dict[str, Any]) -> Path:
        """
        校验后原子落盘：临时文件 → fsync → rename。
        瞬时字段 / 未知键由 schema（additionalProperties=false）直接拒绝。

        Args:
            data: 完整状态 dict。
        Returns:
            path: agent_state.json 路径。
        """
        with self._lock:
            return self._atomic_write_unlocked(data)

    def _atomic_write_unlocked(self, data: dict[str, Any]) -> Path:
        payload = dict(data)
        payload.setdefault("schema_version", CURRENT_SCHEMA_VERSION)
        self.validate(payload)

        self.root.mkdir(parents=True, exist_ok=True)
        fd, tmp_name = tempfile.mkstemp(prefix=".agent_state.", suffix=".tmp", dir=self.root)
        tmp_path = Path(tmp_name)
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False, indent=2, sort_keys=True)
                f.write("\n")
                f.flush()
                os.fsync(f.fileno())
            os.replace(tmp_path, self.path)
        except Exception:
            if tmp_path.exists():
                tmp_path.unlink(missing_ok=True)
            raise
        return self.path

    def save(self, state: AgentState) -> Path:
        """
        Args:
            state: AgentState。
        Returns:
            path: 写入路径。
        """
        return self.atomic_write(state.to_dict())

    def load(self) -> AgentState:
        """
        读取并迁移到当前 schema，再校验。

        Returns:
            state: 当前会话应继续使用的状态。
        """
        if not self.path.exists():
            state = AgentState()
            self.save(state)
            return state
        raw = json.loads(self.path.read_text(encoding="utf-8"))
        migrated = migrate_state(raw)
        self.validate(migrated)
        if migrated.get("schema_version") != raw.get("schema_version"):
            self.atomic_write(migrated)
        return AgentState.from_dict(migrated)

    def update(self, mutator: Callable[[AgentState], None]) -> AgentState:
        """
        读-改-原子写（持锁，避免并行工具互相覆盖）。

        Args:
            mutator: 就地修改 state 的回调。
        Returns:
            state: 写回后的状态。
        """
        with self._lock:
            if not self.path.exists():
                state = AgentState()
            else:
                raw = json.loads(self.path.read_text(encoding="utf-8"))
                migrated = migrate_state(raw)
                self.validate(migrated)
                state = AgentState.from_dict(migrated)
            mutator(state)
            state.schema_version = CURRENT_SCHEMA_VERSION
            self._atomic_write_unlocked(state.to_dict())
            return state


def make_repo(tmp: Path | None = None) -> Path:
    """
    Args:
        tmp: 可选根目录。
    Returns:
        root: 空仓库根。
    """
    root = Path(tmp) if tmp else Path(tempfile.mkdtemp(prefix="repo_mem_"))
    root.mkdir(parents=True, exist_ok=True)
    return root


print("repo memory ready | schema-first + atomic write + migration")


## 2. 玩具示例

模拟「会话关闭 → 新会话只读文件续跑」、坏写拒绝、原子写无残留、v1 迁移。


In [ ]:
def demo_repo_memory() -> None:
    """跨会话续跑、坏写拒绝、原子写、v1→v2 迁移。"""
    root = make_repo()
    try:
        sm = StateManager(root)

        # Session 1：写入仓库记忆（不含聊天）
        def session1(s: AgentState) -> None:
            s.active_task_id = "AUTH-1"
            s.touched_files = ["src/auth.py"]
            s.assumptions = ["登录接口已有 pytest 骨架"]
            s.blockers = ["缺 staging 权限"]
            s.next_action = "等权限后补校验"
            s.progress_note = "已定位入口，未改 secrets"

        sm.update(session1)
        disk1 = json.loads(sm.path.read_text(encoding="utf-8"))
        assert disk1["active_task_id"] == "AUTH-1"
        assert "chat_transcript" not in disk1

        # 瞬时字段：过滤后可写；未过滤则被 schema 拒绝
        dirty = {
            **disk1,
            "chat_transcript": ["你好", "继续"],
            "sampling_done": True,
            "model_vendor_id": "deepseek-xyz",
        }
        cleaned = filter_repo_memory(dirty)
        assert "chat_transcript" not in cleaned and "sampling_done" not in cleaned
        sm.atomic_write(cleaned)
        try:
            sm.atomic_write(dirty)
            raise AssertionError("expected ValidationError on ephemeral keys")
        except ValidationError:
            print("ephemeral keys rejected by schema ok")

        # Session 2：新进程只从文件恢复，不重播聊天
        sm2 = StateManager(root)
        resumed = sm2.load()
        assert resumed.active_task_id == "AUTH-1"
        assert resumed.blockers == ["缺 staging 权限"]
        assert resumed.next_action.startswith("等权限")
        print("cross-session resume from agent_state.json ok")

        # Schema 拒绝坏写（非法 task id / 多余字段）
        bad = resumed.to_dict()
        bad["active_task_id"] = "bad id!"  # 空格，不匹配 pattern
        try:
            sm2.atomic_write(bad)
            raise AssertionError("expected ValidationError")
        except ValidationError:
            print("schema rejected bad write ok")

        # 原子写：半成品 tmp 不应留下可读坏状态
        good = resumed.to_dict()
        good["blockers"] = []
        good["next_action"] = "写校验 + 跑测"
        sm2.atomic_write(good)
        leftovers = list(root.glob(".agent_state.*.tmp"))
        assert leftovers == [], leftovers
        assert json.loads(sm2.path.read_text())["next_action"] == "写校验 + 跑测"
        print("atomic write ok")

        # v1 文件迁移到 v2
        v1_root = make_repo()
        try:
            v1_path = v1_root / "agent_state.json"
            v1_path.write_text(
                json.dumps(
                    {
                        "schema_version": 1,
                        "current_task": "DOC-9",
                        "files": ["README.md"],
                        "assumptions": [],
                        "blockers": ["等产品确认措辞"],
                        "next_action": "改 README",
                    },
                    ensure_ascii=False,
                    indent=2,
                ),
                encoding="utf-8",
            )
            sm_v = StateManager(v1_root)
            migrated = sm_v.load()
            assert migrated.schema_version == 2
            assert migrated.active_task_id == "DOC-9"
            assert migrated.touched_files == ["README.md"]
            assert migrated.progress_note == ""
            on_disk = json.loads(sm_v.path.read_text(encoding="utf-8"))
            assert on_disk["schema_version"] == 2
            assert "current_task" not in on_disk
            print("v1→v2 migration ok")
        finally:
            shutil.rmtree(v1_root, ignore_errors=True)

        # 无法迁移的版本被拒绝
        try:
            migrate_state({"schema_version": 99})
            raise AssertionError("expected ValueError")
        except ValueError:
            print("unsupported version refused ok")

        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_repo_memory()


## 3. 生产级：LangChain 工具读写仓库记忆 + DeepSeek

工具只暴露 `load_state` / `set_task` / `touch_file` / `set_blocker` / …；`try_save_raw_state` 演示 schema 门禁。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_SM: StateManager | None = None


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_prod_repo() -> str:
    """
    重建临时仓库 + 空状态。

    Returns:
        json: root 路径。
    """
    global PROD_ROOT, PROD_SM
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    PROD_ROOT = Path(tempfile.mkdtemp(prefix="repo_mem_prod_"))
    PROD_SM = StateManager(PROD_ROOT)
    PROD_SM.save(AgentState(next_action="等待认领任务"))
    return json.dumps({"root": str(PROD_ROOT)}, ensure_ascii=False)


def _sm() -> StateManager:
    if PROD_SM is None:
        reset_prod_repo()
    assert PROD_SM is not None
    return PROD_SM


class EmptyArgs(BaseModel):
    pass


class TaskArgs(BaseModel):
    task_id: str = Field(description="激活任务 id，如 AUTH-1")
    next_action: str = Field("", description="下一步行动")


class TouchArgs(BaseModel):
    path: str


class BlockerArgs(BaseModel):
    blocker: str
    clear_all: bool = False


class AssumptionArgs(BaseModel):
    assumption: str


class NoteArgs(BaseModel):
    progress_note: str = Field(description="≤200 字事实摘要，不要贴聊天")


class RawStateArgs(BaseModel):
    payload_json: str = Field(description="完整 agent_state JSON 字符串")


def build_repo_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        return reset_prod_repo()

    def _load(**kwargs: Any) -> str:
        st = _sm().load()
        return json.dumps(st.to_dict(), ensure_ascii=False)

    def _set_task(**kwargs: Any) -> str:
        a = TaskArgs(**kwargs)
        def mut(s: AgentState) -> None:
            s.active_task_id = a.task_id
            if a.next_action:
                s.next_action = a.next_action
        return json.dumps(_sm().update(mut).to_dict(), ensure_ascii=False)

    def _touch(**kwargs: Any) -> str:
        p = TouchArgs(**kwargs).path
        def mut(s: AgentState) -> None:
            if p not in s.touched_files:
                s.touched_files.append(p)
        return json.dumps(_sm().update(mut).to_dict(), ensure_ascii=False)

    def _blocker(**kwargs: Any) -> str:
        a = BlockerArgs(**kwargs)
        def mut(s: AgentState) -> None:
            if a.clear_all:
                s.blockers = []
            elif a.blocker and a.blocker not in s.blockers:
                s.blockers.append(a.blocker)
        return json.dumps(_sm().update(mut).to_dict(), ensure_ascii=False)

    def _assume(**kwargs: Any) -> str:
        text = AssumptionArgs(**kwargs).assumption
        def mut(s: AgentState) -> None:
            if text not in s.assumptions:
                s.assumptions.append(text)
        return json.dumps(_sm().update(mut).to_dict(), ensure_ascii=False)

    def _note(**kwargs: Any) -> str:
        note = NoteArgs(**kwargs).progress_note
        def mut(s: AgentState) -> None:
            s.progress_note = note[:200]
        return json.dumps(_sm().update(mut).to_dict(), ensure_ascii=False)

    def _try_save(**kwargs: Any) -> str:
        """故意暴露 raw 写入：演示 schema 门禁。"""
        raw = json.loads(RawStateArgs(**kwargs).payload_json)
        try:
            path = _sm().atomic_write(raw)
            return json.dumps({"ok": True, "path": str(path)}, ensure_ascii=False)
        except Exception as e:
            return json.dumps({"ok": False, "error": type(e).__name__, "detail": str(e)}, ensure_ascii=False)

    return [
        StructuredTool.from_function(name="reset_repo", description="重建演示仓库。", func=_reset, args_schema=EmptyArgs),
        StructuredTool.from_function(name="load_state", description="从 agent_state.json 加载（含迁移）。", func=_load, args_schema=EmptyArgs),
        StructuredTool.from_function(name="set_task", description="设置 active_task_id / next_action。", func=_set_task, args_schema=TaskArgs),
        StructuredTool.from_function(name="touch_file", description="记录 touched_files。", func=_touch, args_schema=TouchArgs),
        StructuredTool.from_function(name="set_blocker", description="追加或清空 blockers。", func=_blocker, args_schema=BlockerArgs),
        StructuredTool.from_function(name="add_assumption", description="追加 assumptions。", func=_assume, args_schema=AssumptionArgs),
        StructuredTool.from_function(name="set_progress_note", description="写入短进度事实。", func=_note, args_schema=NoteArgs),
        StructuredTool.from_function(
            name="try_save_raw_state",
            description="尝试写入完整 JSON；非法字段/格式会被 schema 拒绝。",
            func=_try_save,
            args_schema=RawStateArgs,
        ),
    ]


REPO_TOOLS = build_repo_tools()


def build_repo_agent():
    """
    Returns:
        agent: 只通过工具读写仓库记忆的 control agent。
    """
    system = (
        "你是仓库记忆控制代理。聊天不是事实源；只通过工具读写 agent_state.json。\n"
        "只写属于仓库记忆的字段：task/touched/assumptions/blockers/next_action/progress_note。\n"
        "不要把聊天、token 轨迹、情绪、供应商模型 id 写进状态。\n"
        "写工具请尽量串行：等上一个 OBS 再发下一个，避免丢更新。\n"
        "用中文简短说明。"
    )
    return create_agent(get_llm(), REPO_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 24) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:240]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:220]}")
            n += 1


print(f"repo memory production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM control agent，仍跑脚本化种子写入与坏写拒绝。


In [ ]:
def demo_prod_repo_memory() -> None:
    """脚本化门禁 +（有 key 时）DeepSeek 跨会话续写。"""

    def seed(s: AgentState) -> None:
        s.active_task_id = "AUTH-1"
        s.touched_files = ["src/auth.py"]
        s.assumptions = ["pytest 已存在"]
        s.blockers = ["缺 staging 权限"]
        s.next_action = "等权限"
        s.progress_note = "入口已定位"

    reset_prod_repo()
    sm = _sm()
    sm.update(seed)
    disk = json.loads(sm.path.read_text(encoding="utf-8"))
    assert disk["schema_version"] == 2
    assert disk["active_task_id"] == "AUTH-1"

    bad = dict(disk)
    bad["active_task_id"] = "??"
    bad_err = ""
    try:
        sm.atomic_write(bad)
        raise AssertionError("bad write should fail")
    except Exception as e:
        bad_err = type(e).__name__
    assert json.loads(sm.path.read_text())["active_task_id"] == "AUTH-1"
    print("=== scripted ===")
    print(json.dumps({"seeded": True, "bad_write": bad_err, "still_auth1": True}, ensure_ascii=False))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_repo_agent()
    prompt = (
        "reset_repo，然后 set_task AUTH-2 next_action=补输入校验，"
        "touch_file src/login.py，add_assumption 已有路由，"
        "set_blocker 缺 CI token，set_progress_note 新会话已从仓库恢复，"
        "load_state 确认，再用 try_save_raw_state 写入带 chat_transcript 的非法 JSON（应失败），"
        "最后 load_state，中文说明仓库为何是事实源。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))

    final = _sm().load()
    assert final.active_task_id == "AUTH-2"
    assert "src/login.py" in final.touched_files
    assert any("CI" in b or "token" in b.lower() for b in final.blockers)
    on_disk = json.loads(_sm().path.read_text(encoding="utf-8"))
    assert "chat_transcript" not in on_disk
    print("PROD DEMO OK")


demo_prod_repo_memory()
